# Repeated-measures comparisons within Cluster 2
Marianna Georgiou, Department of Psychiatry, Psychotherapy and Psychosomatics, Faculty of Medicine, RWTH Aachen, Aachen, Germany
**Email:** mariannag43@gmail.com
**Publication:**  Latent subtypes of longitudinal postpartum trajectories for identifying healthy women at risk for affective conditions

Test changes over time within HC Cluster 2 and AD Cluster 2 using Friedman tests and paired Wilcoxon comparisons.


## Input

`files/interm_files/val_merged_rawdata_withlabels.xlsx` for the default `coh = [3]`, or `disc_merged_rawdata_withlabels.xlsx` for `[1, 2]`. Requires ID, CODE values HC/AD/PPD, IDX=1/2, and raw trajectory scores. Discovery EPDS `_y` suffixes are removed; validation expects unsuffixed EPDS totals.

## Outputs

- `files/tables/hc_{cohort}_friedmann_time.xlsx`
- `files/tables/ad_{cohort}_friedmann_time.xlsx`

`{cohort}` is `val` or `disc`. 

## 1. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from scipy.stats import kruskal
from matplotlib.lines import Line2D
from scipy.stats import kruskal
from statsmodels.stats.multitest import multipletests
import matplotlib.gridspec as gridspec

%load_ext dotenv
%dotenv
%load_ext autoreload
%autoreload 2

from pathlib import Path

BASE_DIR = Path.cwd()
INPUT_PATH = BASE_DIR / "files" / "inputs"
OUTPUT_FILES = BASE_DIR / "files" / "interm_files"
OUTPUT_PLOTS = BASE_DIR / "files" / "plots"
OUTPUT_TABLES = BASE_DIR / "files" / "tables"
for directory in (OUTPUT_FILES, OUTPUT_PLOTS, OUTPUT_TABLES):
    directory.mkdir(parents=True, exist_ok=True)

def set_global_plot_style():

    """Apply plotting typography and Seaborn style."""
    sns.set_context("paper")

    plt.rcParams['font.family'] = 'Segoe UI'
    plt.rc('font', size=18)
    plt.rc('axes', titlesize=18)
    plt.rc('axes', labelsize=18)
    plt.rc('xtick', labelsize=15)
    plt.rc('ytick', labelsize=15)
    plt.rc('legend', fontsize=15)
    plt.rc('figure', titlesize=18)

set_global_plot_style()

np.set_printoptions(precision=3, suppress=True)


## 2. Load trajectories and label groups

Map IDX=1/2 to C1/C2, then combine with CODE. Rows are assumed to represent unique participants before repeated-measures reshaping.

In [ ]:
coh=[3]

if coh == [3]:
    data = pd.read_excel(OUTPUT_FILES / "val_merged_rawdata_withlabels.xlsx")
elif coh == [1, 2]:
    data = pd.read_excel(OUTPUT_FILES / "disc_merged_rawdata_withlabels.xlsx")

data['IDX'].value_counts()


In [ ]:
data['Cluster'] = data['IDX'].map({1: 'C1', 2: 'C2'})

data['Group_Label'] = data['CODE'] + ' ' + data['Cluster']


In [ ]:
data[data['Group_Label'] == 'HC C2'].columns
if coh == [1, 2]:
    data.columns = [col.replace('_y', '') if col.startswith('EPDS') else col for col in data.columns]
data[data['Group_Label'] == 'HC C2'].columns
data


## 3. Define measures and available time points

Use T0–T4 except MPAS, which uses T1–T4. Require at least three existing time-point columns and three participants with complete scores across those columns for each measure/group. No imputation is performed.

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import friedmanchisquare, wilcoxon
from statsmodels.stats.multitest import multipletests

symptoms = ['EPDStotal', 'Anhedonie', 'Anxiety', 'Depression', 'Stress', 'Mood', 'MPAS']

timepoints = ['0', '1', '2', '3', '4']
min_timepoints = 3

def get_columns(symptom):
    """Return expected score columns, omitting T0 for MPAS."""
    if symptom == 'MPAS':
        return [f'{symptom} T{tp}' for tp in timepoints if tp != '0']
    elif symptom in ['Stress', 'Mood']:
        return [f'{symptom} T{tp}' for tp in timepoints]
    else:
        return [f'{symptom}_t{tp}' for tp in timepoints]


### Initial all-pairs analysis function

This definition is retained but replaced by the following function before the main export.

In [ ]:
from itertools import combinations
from scipy.stats import friedmanchisquare, wilcoxon
from statsmodels.stats.multitest import multipletests

def analyze_group(group_df, group_label):
    """Run within-group longitudinal tests for each eligible measure."""
    results = {}
    rows=[]
    print(f"\nAnalyzing group: {group_label}")
    print(f"Total subjects: {len(group_df)}")

    for symptom in symptoms:
        print(f"\n--- Symptom: {symptom} ---")
        cols = get_columns(symptom)
        existing_cols = [col for col in cols if col in group_df.columns]
        print(f"Existing columns: {existing_cols}")

        if len(existing_cols) >= min_timepoints:
            data_matrix = group_df[existing_cols].dropna()
            print(f"Data shape after dropping NA: {data_matrix.shape}")
            if data_matrix.shape[0] < 3:
                print("  Skipping: not enough valid subjects.")
                continue

            samples = [data_matrix[col].values for col in existing_cols]
            stat, p = friedmanchisquare(*samples)
            stat = round(stat, 3)
            p = round(p,3)
            friedman_str = f"{stat:.3f}, {p:.3f}"

            print(f"  Friedman test: stat = {stat:.3f}, p = {p:.3f}")
            result = {'friedman_stat': stat, 'friedman_p': p, 'n': len(data_matrix)}
            print(result)

            row = {
                'Lomgitudinal data': symptom,
                'Friedman (test statistic, p-value)': friedman_str,
                'Wilcoxon post-hoc (test statistic, p-value, FDR p-value)': '–'
            }

            if p < 0.05:
                print("  Running full post hoc Wilcoxon tests...")
                comparisons = []
                p_values = []
                stats = []
                labels = []
                for i, j in combinations(range(len(existing_cols)), 2):
                    x = data_matrix[existing_cols[i]]
                    y = data_matrix[existing_cols[j]]
                    stat_, p_ = wilcoxon(x, y)
                    stat_ = round(stat_, 3)
                    p_ = round(p_,3)
                    comparison = f"{existing_cols[i]} vs {existing_cols[j]}"

                    p_values.append(p_)
                    stats.append(stat_)
                    labels.append(f"{existing_cols[i]} vs {existing_cols[j]}")

                    print(f"    {comparison}: Wilcoxon stat = {stat_:.3f}, p = {p_:.4f}")

                # Correct comparisons within this measure/group only.
                reject, pvals_corrected, _, _ = multipletests(p_values, method='fdr_bh')
                posthoc_strs = []

                for label, s, raw_p, fdr_p, sig in zip(labels, stats, p_values, pvals_corrected, reject):
                    s = round(s, 3)
                    raw_p = round(raw_p, 3)
                    fdr_p = round(fdr_p, 3)
                    posthoc_strs.append(f"{label} ({s:.2f}, {raw_p:.3f}, {fdr_p:.3f})")

                if posthoc_strs:
                    row['Wilcoxon post-hoc (test statistic, p-value, FDR p-value)'] = "\n".join(posthoc_strs)

            rows.append(row)
        else:
            print("  Skipping: not enough timepoints available.")

    return pd.DataFrame(rows)


## 4. Friedman and all-pairs Wilcoxon tests

For each group/measure, apply Friedman to complete paired records. If its p-value, rounded to three decimals, is below 0.05, compare every available time-point pair with SciPy’s default paired Wilcoxon test. Wilcoxon p-values are also rounded before BH correction within that group/measure. No correction spans the Friedman tests or different groups/measures. The rounded-p-value behavior and output column spelling are preserved.

In [ ]:
import re
from itertools import combinations
from scipy.stats import friedmanchisquare, wilcoxon
from statsmodels.stats.multitest import multipletests
import pandas as pd

def extract_timepoint(col_name):
    """Extract a standardized T label from the first digit sequence."""
    match = re.search(r'[tT]?\D*?(\d+)', col_name)
    if match:
        return f"T{match.group(1)}"
    else:
        return col_name

def analyze_group(group_df, group_label):
    """Run within-group longitudinal tests for each eligible measure."""
    rows = []
    print(f"\nAnalyzing group: {group_label}")
    print(f"Total subjects: {len(group_df)}")

    for symptom in symptoms:
        print(f"\n--- Symptom: {symptom} ---")
        cols = get_columns(symptom)
        existing_cols = [col for col in cols if col in group_df.columns]
        print(f"Existing columns: {existing_cols}")

        if len(existing_cols) >= min_timepoints:
            data_matrix = group_df[existing_cols].dropna()
            print(f"Data shape after dropping NA: {data_matrix.shape}")
            if data_matrix.shape[0] < 3:
                print("  Skipping: not enough valid subjects.")
                continue

            samples = [data_matrix[col].values for col in existing_cols]
            stat, p = friedmanchisquare(*samples)
            stat = round(stat, 3)
            p = round(p, 3)
            friedman_str = f"{stat:.3f}, {p:.3f}"

            print(f"  Friedman test: stat = {stat:.3f}, p = {p:.3f}")
            result = {'friedman_stat': stat, 'friedman_p': p, 'n': len(data_matrix)}
            print(result)

            row = {
                'Lomgitudinal data': symptom,
                'Friedman (test statistic, p-value)': friedman_str,
                'Wilcoxon post-hoc (test statistic, p-value, FDR p-value)': '–'
            }

            if p < 0.05:
                print("  Running full post hoc Wilcoxon tests...")
                p_values = []
                stats = []
                labels = []

                for i, j in combinations(range(len(existing_cols)), 2):
                    x = data_matrix[existing_cols[i]]
                    y = data_matrix[existing_cols[j]]
                    stat_, p_ = wilcoxon(x, y)
                    stat_ = round(stat_, 3)
                    p_ = round(p_, 3)

                    label_i = extract_timepoint(existing_cols[i])
                    label_j = extract_timepoint(existing_cols[j])
                    comparison = f"{label_i} vs {label_j}"

                    p_values.append(p_)
                    stats.append(stat_)
                    labels.append(comparison)

                    print(f"    {comparison}: Wilcoxon stat = {stat_:.3f}, p = {p_:.3f}")

                # Correct comparisons within this measure/group only.
                reject, pvals_corrected, _, _ = multipletests(p_values, method='fdr_bh')
                posthoc_strs = []

                for label, s, raw_p, fdr_p, sig in zip(labels, stats, p_values, pvals_corrected, reject):
                    s = round(s, 3)
                    raw_p = round(raw_p, 3)
                    fdr_p = round(fdr_p, 3)
                    posthoc_strs.append(f"{label} ({s:.3f}, {raw_p:.3f}, {fdr_p:.3f})")

                if posthoc_strs:
                    row['Wilcoxon post-hoc (test statistic, p-value, FDR p-value)'] = "\n".join(posthoc_strs)

            rows.append(row)
        else:
            print("  Skipping: not enough timepoints available.")

    return pd.DataFrame(rows)


### Run HC C2 and AD C2 analyses and export

In [ ]:
hc_c2 = data[data['Group_Label'] == 'HC C2']
ad_c2 = data[data['Group_Label'] == 'AD C2']

hc_c2_results = analyze_group(hc_c2, 'HC C2')
ad_c2_results = analyze_group(ad_c2, 'AD C2')

if coh == [3]:
    aux = 'val'
else:
    aux = 'disc'

hc_c2_results.to_excel(OUTPUT_TABLES / f'hc_{aux}_friedmann_time.xlsx', index=False)
ad_c2_results.to_excel(OUTPUT_TABLES / f'ad_{aux}_friedmann_time.xlsx', index=False)


### Existing diagnostic lookup

The next cell requests `Wilcoxon post-hoc`, but the exported DataFrame uses the longer descriptive column name. It raises KeyError as supplied, after the Excel exports. This lookup is preserved rather than silently repaired.

In [ ]:
hc_c2_results['Wilcoxon post-hoc'].iloc[0]


## 5. Alternative adjacent-timepoint comparisons

If execution continues past the diagnostic lookup, redefine the analysis to test only consecutive available time points after a significant Friedman result. This version uses unrounded p-values for the gate and within-measure BH correction. It returns dictionaries and overwrites the in-memory results, but does not modify the earlier Excel exports.

In [ ]:
def analyze_group(group_df, group_label):
    """Run within-group longitudinal tests for each eligible measure."""
    results = {}
    print(f"\nAnalyzing group: {group_label}")
    print(f"Total subjects: {len(group_df)}")

    for symptom in symptoms:
        print(f"\n--- Symptom: {symptom} ---")
        cols = get_columns(symptom)
        existing_cols = [col for col in cols if col in group_df.columns]
        print(f"Existing columns: {existing_cols}")

        if len(existing_cols) >= min_timepoints:
            data_matrix = group_df[existing_cols].dropna()
            print(f"Data shape after dropping NA: {data_matrix.shape}")
            if data_matrix.shape[0] < 3:
                print("  Skipping: not enough valid subjects.")
                continue

            samples = [data_matrix[col].values for col in existing_cols]
            stat, p = friedmanchisquare(*samples)
            print(f"  Friedman test: stat = {stat:.3f}, p = {p:.4f}")
            result = {'friedman_stat': stat, 'friedman_p': p, 'n': len(data_matrix)}

            if p < 0.05:
                print("  Running post hoc Wilcoxon tests...")
                comparisons = []
                p_values = []
                stats = []
                for i in range(len(existing_cols) - 1):
                    x = data_matrix[existing_cols[i]]
                    y = data_matrix[existing_cols[i + 1]]
                    stat_, p_ = wilcoxon(x, y)
                    comparison = f'{existing_cols[i]} vs {existing_cols[i + 1]}'
                    comparisons.append(comparison)
                    p_values.append(p_)
                    stats.append(stat_)
                    print(f"    {comparison}: Wilcoxon stat = {stat_:.3f}, p = {p_:.4f}")

                # Correct comparisons within this measure/group only.
                reject, pvals_corrected, _, _ = multipletests(p_values, method='fdr_bh')
                for i, comp in enumerate(comparisons):
                    print(f"    FDR corrected: {comp}, p_corr = {pvals_corrected[i]:.4f}, significant = {reject[i]}")
                result['posthoc'] = list(zip(comparisons, p_values, pvals_corrected, reject))

            results[symptom] = result
        else:
            print("  Skipping: not enough timepoints available.")
    return results

hc_c2 = data[data['Group_Label'] == 'HC C2']
ad_c2 = data[data['Group_Label'] == 'AD C2']

hc_c2_results = analyze_group(hc_c2, 'HC C2')
ad_c2_results = analyze_group(ad_c2, 'AD C2')
